In [1]:
import json
import re
import pandas as pd
import numpy as np

from pathlib import Path

DATA_DIR = Path("../data/cicids")
STIX_FILE = Path("../data/attck/enterprise-attack.json")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "cicids_processed.csv"

In [2]:
# Load ATT&CK techniques
with open(STIX_FILE, "r", encoding="utf-8") as f:
    bundle = json.load(f)

techniques = {}

for obj in bundle.get("objects", []):
    if obj.get("type") != "attack-pattern":
        continue
    if obj.get("revoked", False) or obj.get("deprecated", False):
        continue

    technique_id = None
    for ref in obj.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            technique_id = ref.get("external_id")
            break

    if not technique_id:
        continue

    tactics = []
    for phase in obj.get("kill_chain_phases", []):
        if phase.get("kill_chain_name") == "mitre-attack":
            tactics.append(phase["phase_name"].replace("-", " ").title())

    techniques[technique_id] = {
        "name": obj.get("name", ""),
        "tactic": tactics[0] if tactics else "Unknown"
    }


print(f"Loaded {len(techniques)} ATT&CK techniques")

Loaded 703 ATT&CK techniques


In [3]:
techniques

{'T1055.011': {'name': 'Extra Window Memory Injection',
  'tactic': 'Defense Evasion'},
 'T1053.005': {'name': 'Scheduled Task', 'tactic': 'Execution'},
 'T1205.002': {'name': 'Socket Filters', 'tactic': 'Defense Evasion'},
 'T1560.001': {'name': 'Archive via Utility', 'tactic': 'Collection'},
 'T1021.005': {'name': 'VNC', 'tactic': 'Lateral Movement'},
 'T1047': {'name': 'Windows Management Instrumentation',
  'tactic': 'Execution'},
 'T1113': {'name': 'Screen Capture', 'tactic': 'Collection'},
 'T1027.011': {'name': 'Fileless Storage', 'tactic': 'Defense Evasion'},
 'T1037': {'name': 'Boot or Logon Initialization Scripts',
  'tactic': 'Persistence'},
 'T1557': {'name': 'Adversary-in-the-Middle', 'tactic': 'Credential Access'},
 'T1033': {'name': 'System Owner/User Discovery', 'tactic': 'Discovery'},
 'T1583': {'name': 'Acquire Infrastructure', 'tactic': 'Resource Development'},
 'T1218.011': {'name': 'Rundll32', 'tactic': 'Defense Evasion'},
 'T1613': {'name': 'Container and Resource

In [4]:
# Map CICIDS labels to ATT&CK techniques
# Used as evaluation metadata
raw_map = {
    "FTP-Patator": "T1110.001",
    "FTP Patator": "T1110.001",
    "SSH-Patator": "T1110.001",
    "SSH Patator": "T1110.001",
    "DoS slowloris": "T1499.001",
    "DoS Slowhttptest": "T1499.001",
    "DoS Hulk": "T1499.001",
    "DoS GoldenEye": "T1499.001",
    "Heartbleed": "T1190",
    "Web Attack Brute Force": "T1110.001",
    "Web Attack XSS": "T1059.007",
    "Web Attack Sql Injection": "T1190",
    "Infiltration": "T1105",
    "Bot": "T1071.001",
    "PortScan": "T1046",
    "DDoS": "T1498.001",
    "BENIGN": None,
}

label_map = {}

for label, technique_id in raw_map.items():
    if technique_id is None:
        label_map[label] = {
            "technique_id": "BENIGN",
            "technique_name": "Benign Traffic",
            "tactic": "Benign"
        }
    else:
        info = techniques.get(technique_id, {"name": "Unknown", "tactic": "Unknown"})
        label_map[label] = {
            "technique_id": technique_id,
            "technique_name": info["name"],
            "tactic": info["tactic"]
        }

label_map

{'FTP-Patator': {'technique_id': 'T1110.001',
  'technique_name': 'Password Guessing',
  'tactic': 'Credential Access'},
 'FTP Patator': {'technique_id': 'T1110.001',
  'technique_name': 'Password Guessing',
  'tactic': 'Credential Access'},
 'SSH-Patator': {'technique_id': 'T1110.001',
  'technique_name': 'Password Guessing',
  'tactic': 'Credential Access'},
 'SSH Patator': {'technique_id': 'T1110.001',
  'technique_name': 'Password Guessing',
  'tactic': 'Credential Access'},
 'DoS slowloris': {'technique_id': 'T1499.001',
  'technique_name': 'OS Exhaustion Flood',
  'tactic': 'Impact'},
 'DoS Slowhttptest': {'technique_id': 'T1499.001',
  'technique_name': 'OS Exhaustion Flood',
  'tactic': 'Impact'},
 'DoS Hulk': {'technique_id': 'T1499.001',
  'technique_name': 'OS Exhaustion Flood',
  'tactic': 'Impact'},
 'DoS GoldenEye': {'technique_id': 'T1499.001',
  'technique_name': 'OS Exhaustion Flood',
  'tactic': 'Impact'},
 'Heartbleed': {'technique_id': 'T1190',
  'technique_name': '

In [5]:
# helper functions
# Normalise labels by removing non-ASCII chars, replacing dashes with spaces, and stripping whitespace
def normalise_label(label):
    """Cleans label text so mapping is consistent."""
    if not isinstance(label, str):
        return "BENIGN"
    label = re.sub(r"[^\x00-\x7F]+", "", label)
    label = re.sub(r"\s*[-–—]\s*", " ", label)
    label = re.sub(r"\s+", " ", label).strip()
    return label


def to_int(value, default=0):
    try:
        return int(float(value))
    except Exception:
        return default


def to_float(value, default=0.0):
    try:
        return float(value)
    except Exception:
        return default


def build_alert_text(row):
    """
    Builds a neutral, descriptive text description of a CIC flow.
    """
    dst_port = to_int(row.get("Destination Port", 0))
    duration = to_int(row.get("Flow Duration", 0))
    fwd_pkts = to_int(row.get("Total Fwd Packets", 0))
    bwd_pkts = to_int(row.get("Total Backward Packets", 0))
    flow_bps = round(to_float(row.get("Flow Bytes/s", 0.0)), 2)

    syn_flag = to_int(row.get("SYN Flag Count", 0))
    rst_flag = to_int(row.get("RST Flag Count", 0))
    ack_flag = to_int(row.get("ACK Flag Count", 0))

    port_map = {
        21: "FTP service",
        22: "SSH service",
        23: "Telnet service",
        25: "SMTP service",
        53: "DNS service",
        80: "HTTP service",
        443: "HTTPS service",
        3389: "RDP service",
        8080: "HTTP-alt service",
    }

    service = port_map.get(dst_port, f"unknown service on port {dst_port}")

    flags = []
    if syn_flag > 0:
        flags.append("SYN")
    if rst_flag > 0:
        flags.append("RST")
    if ack_flag > 0:
        flags.append("ACK")
    flag_text = ", ".join(flags) if flags else "none"

    # Light descriptive behavior hints
    behavior = []

    if duration < 100:
        behavior.append("very short connection")
    elif duration > 1_000_000:
        behavior.append("long duration")

    total_pkts = fwd_pkts + bwd_pkts
    if total_pkts <= 2:
        behavior.append("low packet count")
    elif total_pkts > 1000:
        behavior.append("high packet count")

    if flow_bps > 1_000_000:
        behavior.append("high volume traffic")

    behavior_text = ". ".join(behavior) if behavior else "observed traffic pattern"

    text = (
        f"Network flow to {service}. "
        f"{behavior_text}. "
        f"Duration {duration} microseconds. "
        f"Forward packets {fwd_pkts}. "
        f"Backward packets {bwd_pkts}. "
        f"Flow bytes per second {flow_bps}. "
        f"TCP flags: {flag_text}."
    )

    return text

In [6]:
# Map a CICIDS label to its corresponding ATT&CK technique info
def map_field(label, field):
    info = label_map.get(label)
    if info is None:
        if field == "technique_id":
            return "UNMAPPED"
        return "Unknown"
    return info[field]

In [7]:
# List CSV files in the data directory
csv_files = sorted(DATA_DIR.glob("*.csv"))

print("Folder:", DATA_DIR.resolve())
print("CSV files found:", len(csv_files))
for f in csv_files:
    print("-", f.name)

Folder: /home/amenadiel/RAG-Based-Incident-Reporting-on-SIEM-Log-Streams/data/cicids
CSV files found: 8
- Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
- Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
- Friday-WorkingHours-Morning.pcap_ISCX.csv
- Monday-WorkingHours.pcap_ISCX.csv
- Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
- Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
- Tuesday-WorkingHours.pcap_ISCX.csv
- Wednesday-workingHours.pcap_ISCX.csv


In [8]:
# Load and process each CSV file
all_frames = []

for filepath in csv_files:
    name = filepath.name.lower()

    if "tuesday" in name:
        day = "Tuesday"
    elif "wednesday" in name:
        day = "Wednesday"    
    elif "thursday" in name:
        day = "Thursday"
    elif "friday" in name:
        day = "Friday"
    else:
        print(f"Skipping {filepath.name}")
        continue

    print(f"\nLoading {filepath.name}")

    df = pd.read_csv(filepath, low_memory=False, encoding="latin-1")
    df.columns = df.columns.str.strip()

    # Normalise labels first
    df["Label"] = df["Label"].apply(normalise_label)

    # Keep all attacks, sample 10% benign for balance
    attacks = df[df["Label"] != "BENIGN"].copy()
    benign = df[df["Label"] == "BENIGN"].sample(frac=0.10, random_state=42).copy()
    df = pd.concat([attacks, benign], ignore_index=True)

    # Add ATT&CK metadata
    df["attck_technique_id"] = df["Label"].apply(lambda x: map_field(x, "technique_id"))
    df["attck_technique_name"] = df["Label"].apply(lambda x: map_field(x, "technique_name"))
    df["attck_tactic"] = df["Label"].apply(lambda x: map_field(x, "tactic"))
    df["day"] = day

    # Clean numeric values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

    # Build alert text
    df["alert_text"] = df.apply(build_alert_text, axis=1)

    all_frames.append(df)

print(f"\nLoaded dataframes: {len(all_frames)}")


Loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

Loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv

Loading Friday-WorkingHours-Morning.pcap_ISCX.csv
Skipping Monday-WorkingHours.pcap_ISCX.csv

Loading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv

Loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv

Loading Tuesday-WorkingHours.pcap_ISCX.csv

Loading Wednesday-workingHours.pcap_ISCX.csv

Loaded dataframes: 7


In [9]:
#combine columns
combined = pd.concat(all_frames, ignore_index=True)

print("All columns in combined dataframe:\n")
print(list(combined.columns))

All columns in combined dataframe:

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count

In [10]:
# columns to keep 
keep_cols = [
    "Destination Port",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Flow Bytes/s",
    "SYN Flag Count",
    "RST Flag Count",
    "ACK Flag Count",
    "Label",
    "attck_technique_id",
    "attck_technique_name",
    "attck_tactic",
    "day",
    "alert_text"
]

keep_cols = [c for c in keep_cols if c in combined.columns]
combined = combined[keep_cols].copy()
combined.reset_index(drop=True, inplace=True)
combined.index.name = "alert_id"

print(combined.shape)
combined.head()

(731965, 14)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Flow Bytes/s,SYN Flag Count,RST Flag Count,ACK Flag Count,Label,attck_technique_id,attck_technique_name,attck_tactic,day,alert_text
alert_id,,,,,,,,,,,,,,
0,80,1293792,3,7,8991.398927,0,0,0,DDoS,T1498.001,Direct Network Flood,Impact,Friday,Network flow to HTTP service. long duration. D...
1,80,4421382,4,0,5.428167,0,0,1,DDoS,T1498.001,Direct Network Flood,Impact,Friday,Network flow to HTTP service. long duration. D...
2,80,1083538,3,6,10730.588130,0,0,0,DDoS,T1498.001,Direct Network Flood,Impact,Friday,Network flow to HTTP service. long duration. D...
3,80,80034360,8,4,145.649943,0,0,1,DDoS,T1498.001,Direct Network Flood,Impact,Friday,Network flow to HTTP service. long duration. D...
4,80,642654,3,6,18101.497850,0,0,0,DDoS,T1498.001,Direct Network Flood,Impact,Friday,Network flow to HTTP service. observed traffic...


In [11]:
combined["alert_text"].head(5)

alert_id
0    Network flow to HTTP service. long duration. D...
1    Network flow to HTTP service. long duration. D...
2    Network flow to HTTP service. long duration. D...
3    Network flow to HTTP service. long duration. D...
4    Network flow to HTTP service. observed traffic...
Name: alert_text, dtype: str

In [12]:
# Save to CSV
combined.to_csv(OUTPUT_FILE)
print(f"Saved to {OUTPUT_FILE}")

Saved to ../data/processed/cicids_processed.csv
